# LangSmith + Ragas로 RAG 평가하기

- RAG는 답변만 보는 것이 아니라 **검색된 문서가 맞는지**, **답변이 문서에 근거하는지**, **정답과 얼마나 가까운지**를 함께 봐야 합니다.
- LangSmith는 실행 기록, Dataset, Experiment를 관리합니다.
- Ragas는 RAG 결과를 여러 metric으로 자동 평가합니다.

### 전체 흐름
```text
작은 문서 집합 준비
→ Retriever + RAG 체인 만들기
→ 평가용 질문/정답 만들기
→ Ragas로 직접 평가
→ LangSmith Dataset/Experiment에서 Ragas 평가 실행
```


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.

In [1]:
# 필요한 라이브러리 설치
# uv add -U langchain langchain-core langchain-openai langchain-chroma langsmith ragas python-dotenv pandas numpy
# 또는
# pip install -U langchain langchain-core langchain-openai langchain-chroma langsmith ragas python-dotenv pandas numpy

## (2) API Key 설정

`.env` 파일에 아래 값이 있으면 자동으로 불러옵니다.

```text
OPENAI_API_KEY=...

LANGSMITH_TRACING=...
LANGSMITH_ENDPOINT=...
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=...
```


In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("LANGSMITH_API_KEY:", "있음" if os.getenv("LANGSMITH_API_KEY") else "없음")
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))

OPENAI_API_KEY: 있음
LANGSMITH_API_KEY: 있음
LANGSMITH_TRACING: true


## 2. RAG 평가에서 보는 것

- Ragas metric은 아래 관점을 점수로 만들어줌
- LangSmith는 그 점수와 실행 기록을 Dataset, Experiment 단위로 관리함

| 평가 관점 | 질문 |
|---|---|
| 검색 품질 | 질문에 필요한 문서를 잘 가져왔나? |
| 근거 충실성 | 답변이 검색 문서 안의 내용에 근거하나? |
| 정답성 | 사람이 준비한 기준 답변과 비슷한가? |
| 질문 관련성 | 답변이 질문에 직접 답하고 있나? |


## 3. 실습 문서 준비

- 외부 파일 없이 바로 실행할 수 있도록 간단한 사내 규정 문서로 실습

In [3]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수자는 사내 시스템 접근 권한이 제한된다.",
        metadata={"category": "security", "source": "security_guide"},
    ),
    Document(
        page_content="법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 한다. 지연 등록 시 팀장 승인이 필요하다.",
        metadata={"category": "expense", "source": "expense_policy"},
    ),
    Document(
        page_content="개인정보가 포함된 문서는 외부 공유 전에 반드시 마스킹해야 한다. 고객명, 연락처, 주민등록번호는 마스킹 대상이다.",
        metadata={"category": "privacy", "source": "privacy_policy"},
    ),
    Document(
        page_content="장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함해 서비스 복구 후 24시간 이내에 작성한다.",
        metadata={"category": "incident", "source": "incident_policy"},
    ),
    Document(
        page_content="재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 받은 뒤 근무 장소를 명시해야 한다.",
        metadata={"category": "hr", "source": "hr_policy"},
    ),
]

print(f"문서 수: {len(documents)}")
print(documents[0])

문서 수: 5
page_content='신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수자는 사내 시스템 접근 권한이 제한된다.' metadata={'category': 'security', 'source': 'security_guide'}


## 4. Chroma Retriever 만들기

In [4]:
from uuid import uuid4

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

# 노트북 셀을 반복 실행해도 기존 collection과 충돌하지 않도록 이름을 매번 새로 만듭니다.
collection_name = f"ragas_demo_{uuid4().hex[:8]}"

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=collection_name,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

question = "법인카드 영수증은 언제까지 등록해야 하나요?"
for doc in retriever.invoke(question):
    print(doc.metadata)
    print(doc.page_content)
    print()

{'source': 'expense_policy', 'category': 'expense'}
법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 한다. 지연 등록 시 팀장 승인이 필요하다.

{'source': 'hr_policy', 'category': 'hr'}
재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 받은 뒤 근무 장소를 명시해야 한다.



## 5. 간단한 RAG 체인 만들기

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "너는 사내 규정 Q&A assistant다. 반드시 제공된 참고 문서만 근거로 답하고, 문서에 없으면 모른다고 답한다.",
    ),
    (
        "user",
        "질문: {question}\n\n참고 문서:\n{context}",
    ),
])

answer_chain = prompt | llm | StrOutputParser()


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[{i}] source={doc.metadata.get('source')}\n{doc.page_content}"
        for i, doc in enumerate(docs, start=1)
    )


def rag_answer(question: str) -> dict:
    context_docs = retriever.invoke(question)
    answer = answer_chain.invoke({"question": question, "context": format_docs(context_docs)})
    return {"answer": answer, "contexts": context_docs}


In [6]:
sample = rag_answer("개인정보 문서는 외부 공유 전에 어떻게 해야 하나요?")

print("답변:")
print(sample["answer"])

print("\n검색 문서:")
for doc in sample["contexts"]:
    print("-", doc.metadata["source"], doc.page_content[:80])

답변:
개인정보가 포함된 문서는 외부 공유 전에 반드시 마스킹해야 합니다. 특히 고객명, 연락처, 주민등록번호는 마스킹 대상입니다.

검색 문서:
- privacy_policy 개인정보가 포함된 문서는 외부 공유 전에 반드시 마스킹해야 한다. 고객명, 연락처, 주민등록번호는 마스킹 대상이다.
- security_guide 신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수자는 사내 시스템 접근 권한이 제한된다.


## 6. 평가 데이터 만들기

- 실무에서는 운영 로그, 사람이 만든 golden set, Ragas 합성 데이터 등을 사용할 수 있음
- 여기서는 가장 이해하기 쉬운 수동 평가셋을 사용

In [7]:
eval_questions = [
    {
        "question": "신규 입사자는 보안 교육을 언제까지 들어야 하나요?",
        "reference": "신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 합니다.",
    },
    {
        "question": "법인카드 영수증 등록 기한은 언제인가요?",
        "reference": "법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 합니다.",
    },
    {
        "question": "장애 보고서에는 어떤 내용이 들어가야 하나요?",
        "reference": "장애 보고서에는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책이 포함되어야 합니다.",
    },
    {
        "question": "재택근무는 언제까지 신청해야 하나요?",
        "reference": "재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 합니다.",
    },
]

len(eval_questions)

4

In [8]:
import pandas as pd

ragas_rows = []

# 평가 질문 목록을 하나씩 꺼내서 RAG 시스템에 질문
for item in eval_questions:
    # 현재 질문을 RAG 파이프라인에 넣고 답변과 검색 문서를 받음
    result = rag_answer(item["question"])

    # Ragas가 요구하는 평가 데이터 형식에 맞게 한 컬럼 추가
    ragas_rows.append(
        {
            "user_input": item["question"],             # 사용자가 입력한 질문
            "retrieved_contexts": [doc.page_content for doc in result["contexts"]],     # 본문 내용
            "response": result["answer"],               # RAG가 생성한 답변
            "reference": item["reference"],             # 정답 또는 기준 답변. Ragas 평가에서 response
        }
    )

pd.DataFrame(ragas_rows)

,user_input,retrieved_contexts,response,reference
0,신규 입사자는 보안 교육을 언제까지 들어야 하나요?,"[신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수자는 ...",신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 합니다. 교육을 이수하지 ...,신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 합니다.
1,법인카드 영수증 등록 기한은 언제인가요?,[법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 한다....,법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 합니다....,법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 합니다.
2,장애 보고서에는 어떤 내용이 들어가야 하나요?,"[장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함해 서비...","장애 보고서에는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책이 포함되어야 ...","장애 보고서에는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책이 포함되어야 ..."
3,재택근무는 언제까지 신청해야 하나요?,"[재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 ...","재택근무는 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 받은 뒤...",재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 합니다.


## 7. Ragas로 직접 평가하기

- Ragas는 `user_input`, `retrieved_contexts`, `response`, `reference` 같은 컬럼을 사용해 RAG 품질을 계산

    - `LLMContextRecall`: 정답에 필요한 정보가 검색 문서에 들어있는가
    - `Faithfulness`: 답변이 검색 문서에 근거하는가
    - `FactualCorrectness`: 기준 답변과 사실적으로 얼마나 가까운가

In [9]:
from openai import OpenAI

# ragas import 전에 임시 호환 코드
def patch_ragas_langchain_vertex_import() -> None:
    """
    ragas 0.4.x와 langchain-community 0.4.x 조합에서
    ragas import 시점에 없는 VertexAI 경로를 찾는 문제를 우회합니다.

    이 코드는 VertexAI를 사용하는 코드가 아닙니다.
    OpenAI만 쓰는 실습에서 ragas import가 실패하지 않게 하는 임시 호환 코드입니다.
    """
    import importlib.util
    import sys
    import types

    module_name = "langchain_community.chat_models.vertexai"

    if importlib.util.find_spec(module_name) is not None:
        return

    vertexai_module = types.ModuleType(module_name)

    class ChatVertexAI:
        def __init__(self, *args, **kwargs):
            raise ImportError(
                "ChatVertexAI is not installed. This notebook uses OpenAI only, "
                "so VertexAI should not be instantiated."
            )

    vertexai_module.ChatVertexAI = ChatVertexAI
    sys.modules[module_name] = vertexai_module

patch_ragas_langchain_vertex_import()

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.llms import llm_factory
from ragas.metrics._context_recall import LLMContextRecall
from ragas.metrics._faithfulness import Faithfulness
from ragas.metrics._factual_correctness import FactualCorrectness

In [10]:
# 앞에서 만든 평가 데이터 리스트를 Ragas 전용 데이터셋으로 변환
evaluation_dataset = EvaluationDataset.from_list(ragas_rows)

In [11]:
# Ragas 평가에 사용할 LLM을 만듭니다. 여기선는 gpt-4.1-mini가 evaluator 역할을 함
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
evaluator_llm = llm_factory('gpt-4.1-mini', client=openai_client)

In [12]:
ragas_metrics = [
    LLMContextRecall(),
    Faithfulness(),
    FactualCorrectness()
]

ragas_result = evaluate(
    dataset=evaluation_dataset,
    metrics=ragas_metrics,
    llm=evaluator_llm
)

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

In [13]:
ragas_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,factual_correctness(mode=f1)
0,신규 입사자는 보안 교육을 언제까지 들어야 하나요?,"[신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수자는 ...",신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 합니다. 교육을 이수하지 ...,신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 합니다.,1.0,1.0,0.67
1,법인카드 영수증 등록 기한은 언제인가요?,[법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 한다....,법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 합니다....,법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 합니다.,1.0,1.0,0.67
2,장애 보고서에는 어떤 내용이 들어가야 하나요?,"[장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함해 서비...","장애 보고서에는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책이 포함되어야 ...","장애 보고서에는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책이 포함되어야 ...",1.0,1.0,0.89
3,재택근무는 언제까지 신청해야 하나요?,"[재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 ...","재택근무는 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 받은 뒤...",재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 합니다.,1.0,1.0,0.50


## 8. 평가 결과 해석하기

- 점수가 낮을 때 바로 모델을 바꾸기보다 어느 단계가 문제인지 먼저 나눠 봅니다.

| 낮은 metric | 먼저 의심할 부분 |
|---|---|
| context_recall | 검색기가 필요한 문서를 못 가져옴 |
| faithfulness | LLM이 문서 밖 내용을 추가함 |
| factual_correctness | 답변이 기준 답변과 다름 |
| answer_relevancy / response_relevancy | 질문에 직접 답하지 않음 |

In [14]:
'''
나온 결과 해석하기

- context_recall 1.0
: 질문에 답하는데 필요한 정보가 검색된 문서 안에 충분히 있었다는 뜻.
retriever가 필요한 근거 문서 잘 가져왔다는 의미.

- faithfulness 1.0
: 생성된 답변이 검색 문서에 잘 근거했다는 뜻. 즉 LLM이 문서에 없는 내용을 거의 지어내지 않았다는 의미.
-> 낮으면 환각 가능성이 커짐. 위험할 수 있음.

- factual_correctness(mode=f1) 0.67
: RAG 답변이 사람이 준비한 기준 정답(reference)과 사실적으로 얼마나 일치하는지를 본 점수.
기준 답변과 표현이나 세부 내용에서 약간 차이가 있다는 뜻
-> reference가 짧거나 불완전하면 실제로 맞는 답도 낮게 나올 수 있음.

'''

'\n나온 결과 해석하기\n\n- context_recall 1.0\n: 질문에 답하는데 필요한 정보가 검색된 문서 안에 충분히 있었다는 뜻.\nretriever가 필요한 근거 문서 잘 가져왔다는 의미.\n\n- faithfulness 1.0\n: 생성된 답변이 검색 문서에 잘 근거했다는 뜻. 즉 LLM이 문서에 없는 내용을 거의 지어내지 않았다는 의미.\n-> 낮으면 환각 가능성이 커짐. 위험할 수 있음.\n\n- factual_correctness(mode=f1) 0.67\n: RAG 답변이 사람이 준비한 기준 정답(reference)과 사실적으로 얼마나 일치하는지를 본 점수.\n기준 답변과 표현이나 세부 내용에서 약간 차이가 있다는 뜻\n-> reference가 짧거나 불완전하면 실제로 맞는 답도 낮게 나올 수 있음.\n\n'

## 9. LangSmith Dataset에 평가셋 저장하기

- LangSmith Dataset에 질문과 기준 답변을 저장하면 같은 평가셋으로 여러 RAG 버전을 반복 비교할 수 있음

In [15]:
from datetime import datetime
from langsmith import  Client

client = Client()
dataset_name = f"FYP-{datetime.now():%Y%m%d-%H%M%S}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="RAG 평가 실습용 작은 한국어 Dataset"
)

examples = [
    {
        "inputs": {"question": item["question"]},
        "outputs": {"reference": item["reference"]},
        "metadata": {"source": "manual"},
    }
    for item in eval_questions
]

client.create_examples(dataset_id=dataset.id,
                       examples=examples)

print(dataset_name)

FYP-20260623-174835


## 10. LangSmith Experiment에서 Ragas 평가 실행하기

- `predict()`는 LangSmith가 평가할 대상 RAG 함수
- `ragas_evaluator()`는 각 실행 결과를 Ragas metric으로 채점

In [16]:
from langsmith import traceable

# predict 함수가 실행될 때마다 랭스미스에 trace가 남도록 설정
@traceable(name='simple_rag_for_ragas')
def predict(inputs: dict) -> dict:
    result = rag_answer(inputs["question"])

    # 랭스미스 평가에서 사용할 출력 형식으로 반환
    return {
        "answer": result["answer"],
        "contexts": [doc.page_content for doc in result["contexts"]],
    }

predict({"question": "법인카드 영수증은 언제까지 등록해야 하나요?"})

{'answer': '법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 합니다. 만약 지연 등록할 경우에는 팀장 승인이 필요합니다.',
 'contexts': ['법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 한다. 지연 등록 시 팀장 승인이 필요하다.',
  '재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 받은 뒤 근무 장소를 명시해야 한다.']}

In [17]:
def ragas_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> list[dict]:
    from langchain_core.callbacks import CallbackManager
    from ragas import SingleTurnSample
    from ragas.metrics import FactualCorrectness, Faithfulness
    from ragas.run_config import RunConfig

    # inputs : Dataset에 저장된 입력값
    # outputs : predict() 함수가 반환한 결과값
    # reference_outputs : 기준 정답

    # Ragas가 요구하는 평가 데이터 형식으로 한 건의 row를 만듦
    row = {
        "user_input": inputs["question"],                 # 사용자 질문
        "retrieved_contexts": outputs["contexts"],         # RAG가 검색한 참고 문서들
        "response": outputs["answer"],                     # RAG가 생성한 답변
        "reference": reference_outputs["reference"],       # 사람이 준비한 기준 정답
    }

    # LangSmith evaluator 안에서 Ragas evaluate()를 다시 호출하면
    # Ragas 0.4.x의 trace parsing이 LangSmith evaluator trace와 충돌할 수 있습니다.
    # 그래서 한 건짜리 sample을 metric별로 직접 채점합니다.
    def isolated_callbacks() -> CallbackManager:
        return CallbackManager(handlers=[])

    sample = SingleTurnSample(**row)
    scores = {}
    for metric in [Faithfulness(llm=evaluator_llm), FactualCorrectness(llm=evaluator_llm)]:
        metric.init(RunConfig())
        scores[metric.name] = metric.single_turn_score(
            sample,
            callbacks=isolated_callbacks(),
        )

    # LangSmith evaluator는 {"key": 평가명, "score": 점수} 형태 반환해야 함
    # 여러 지표를 남기고 싶으면 list[dict]로 여러 개를 반환함
    return [
        {"key": "ragas_faithfulness", "score": float(scores.get("faithfulness", 0.0))},
        {"key": "ragas_factual_correctness", "score": float(scores.get("factual_correctness", 0.0))},
    ]

In [18]:
# 이름이 Ragas의 evaluate와 겹치므로 langsmith_evaluate 별칭 사용
from langsmith import evaluate as langsmith_evaluate

experiment_results = langsmith_evaluate(
    predict,
    data=dataset_name,
    evaluators=[ragas_evaluator],
    experiment_prefix="FYP",
    max_concurrency=1
)

experiment_results

View the evaluation results for experiment: 'FYP-6d5e0d1e' at:
https://smith.langchain.com/o/ec56a1cb-7203-41a6-988a-64b1bc2c8bd3/datasets/f5299824-54c7-42a9-820b-ec3e1f9deccb/compare?selectedSessions=b7de86e3-5352-48dc-9487-fd7b03ff334a




0it [00:00, ?it/s]

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_28004\1132111098.py:4: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import FactualCorrectness, Faithfulness
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_28004\1132111098.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import FactualCorrectness, Faithfulness
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_28004\1132111098.py:4: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collection

,inputs.question,outputs.answer,outputs.contexts,error,reference.reference,feedback.ragas_faithfulness,feedback.ragas_factual_correctness,execution_time,example_id,id
0,장애 보고서에는 어떤 내용이 들어가야 하나요?,"장애 보고서에는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책이 포함되어야 ...","[장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함해 서비...",None,"장애 보고서에는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책이 포함되어야 ...",1.0,0.89,1.227834,2e79f525-f2f9-4901-b152-71accb399c5b,019ef3aa-e21b-7831-a2ba-28b6a06550ea
1,신규 입사자는 보안 교육을 언제까지 들어야 하나요?,신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 합니다.,"[신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수자는 ...",None,신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 합니다.,1.0,1.00,1.084017,53458677-17c9-4db6-8a46-0fe34dfbbc35,019ef3aa-e6e8-7a40-a3f4-a79fa6cbe78c
2,재택근무는 언제까지 신청해야 하나요?,"재택근무는 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 받은 뒤...","[재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 하며, 부서장의 승인을 ...",None,재택근무 신청은 최소 하루 전까지 근태 시스템에 등록해야 합니다.,1.0,0.50,1.488253,cc66b538-bdf6-4be5-bb84-b400aa6b5b2f,019ef3aa-eb24-7e93-9419-dfc2be243123
3,법인카드 영수증 등록 기한은 언제인가요?,법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 합니다....,[법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 한다....,None,법인카드 영수증은 결제일 기준 5영업일 이내에 경비 처리 시스템에 등록해야 합니다.,1.0,0.67,1.224218,d72c0486-5743-4db5-b5e1-0cddd12457e1,019ef3aa-f0f5-7423-8495-1396b04af2e9


## 11. 정리

- Ragas는 RAG 결과를 `검색`, `근거`, `정답성`, `관련성` 관점으로 자동 평가합니다.
- LangSmith는 평가셋과 실험 결과를 저장해서 RAG 버전별 비교를 쉽게 만듭니다.
- 낮은 점수는 모델 문제일 수도 있지만, 보통은 검색기 설정, 청크 크기, 프롬프트, 기준 답변 품질을 함께 봐야 합니다.
- 작은 평가셋으로 빠르게 반복한 뒤, 운영 로그와 사람 검수 데이터로 Dataset을 키워가는 방식이 실무적입니다.

## [실습]

1. `retriever = vectorstore.as_retriever(search_kwargs={"k": 1})`로 바꾼 뒤 Ragas 점수를 비교합니다.
2. `k=3`으로 바꾼 뒤 `faithfulness`와 `context_recall`이 어떻게 변하는지 확인합니다.
3. 문서에 없는 질문을 평가셋에 추가하고 RAG가 모른다고 답하는지 확인합니다.
4. `prompt`에서 "문서에 없으면 모른다고 답한다" 문장을 제거하고 hallucination이 늘어나는지 봅니다.
5. LangSmith 화면에서 experiment를 열고 각 질문의 trace와 Ragas 점수를 확인합니다.